# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(readxl)
library(ggplot2)
library(viridis)
library(dplyr)
library(glue)
library(tidyr)

# Load Data

In [6]:
# List all DEG files
files <- list.files(path = "06_DEG_GO", pattern = "\\.xlsx$", full.names = TRUE)

# Keep only comparisons at same time point
files <- files[grepl("(_D\\d+)_vs_.*(_D\\d+)_", files) &
                        sub(".*(_D\\d+)_vs_.*(_D\\d+)_.*", "\\1", files) ==
                        sub(".*_vs_.*(_D\\d+)_.*", "\\1", files)]

# View
print(files)

[1] "06_DEG_GO/R270X_D18_vs_WT_D18_results_enrichr_results.xlsx"
[2] "06_DEG_GO/R270X_D46_vs_WT_D46_results_enrichr_results.xlsx"
[3] "06_DEG_GO/R270X_D90_vs_WT_D90_results_enrichr_results.xlsx"


# Combined Dot Plot

In [29]:
# List of GO databases
databases <- c("GO_Biological_Process_2025", "GO_Cellular_Component_2025", "GO_Molecular_Function_2025", "KEGG_2021_Human")

In [30]:
# Initialize an empty dataframe for all GO data
all_data <- data.frame()

In [31]:
# Iterate over each module and read in the corresponding Excel file
for (file in files) {
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file, sheet = database)
      
      if (nrow(df) == 0) {
        next  # Skip to the next database if no data
      }
      
      # Select the required columns and add module and database information
      df <- df %>%
        filter(P.value <= 0.05) %>%
        select(Term, P.value, Odds.Ratio) %>%
        mutate(Module = file, Database = database) %>%
        arrange(P.value)
      
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Display the combined dataframe
head(all_data)

,Term,P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Intraciliary Transport Involved in Cilium Assembly (GO:0035735),0.03837623,30.79845,06_DEG_GO/R270X_D18_vs_WT_D18_results_enrichr_results.xlsx,GO_Biological_Process_2025
2,Nucleobase-Containing Small Molecule Interconversion (GO:0015949),0.04462864,25.66408,06_DEG_GO/R270X_D18_vs_WT_D18_results_enrichr_results.xlsx,GO_Biological_Process_2025
3,Nucleic Acid Biosynthetic Process (GO:0141187),0.02254599,55.29167,06_DEG_GO/R270X_D46_vs_WT_D46_results_enrichr_results.xlsx,GO_Biological_Process_2025
4,Tetrahydrofolate Interconversion (GO:0035999),0.02699450,44.23111,06_DEG_GO/R270X_D46_vs_WT_D46_results_enrichr_results.xlsx,GO_Biological_Process_2025
5,Maintenance of Synapse Structure (GO:0099558),0.03583154,31.59048,06_DEG_GO/R270X_D46_vs_WT_D46_results_enrichr_results.xlsx,GO_Biological_Process_2025
6,Tetrahydrofolate Metabolic Process (GO:0046653),0.04458920,24.56790,06_DEG_GO/R270X_D46_vs_WT_D46_results_enrichr_results.xlsx,GO_Biological_Process_2025


In [32]:
# Format comparison name
all_data$Module <- gsub("06_DEG_GO/", "", all_data$Module)
all_data$Module <- gsub("_results_enrichr_results.xlsx", "", all_data$Module)
all_data$Module <- gsub("_CtO_", " ", all_data$Module)
all_data$Module <- gsub("_", " ", all_data$Module)
all_data$Module <- gsub("MECP2", "", all_data$Module)

# View
head(all_data)

,Term,P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Intraciliary Transport Involved in Cilium Assembly (GO:0035735),0.03837623,30.79845,R270X D18 vs WT D18,GO_Biological_Process_2025
2,Nucleobase-Containing Small Molecule Interconversion (GO:0015949),0.04462864,25.66408,R270X D18 vs WT D18,GO_Biological_Process_2025
3,Nucleic Acid Biosynthetic Process (GO:0141187),0.02254599,55.29167,R270X D46 vs WT D46,GO_Biological_Process_2025
4,Tetrahydrofolate Interconversion (GO:0035999),0.02699450,44.23111,R270X D46 vs WT D46,GO_Biological_Process_2025
5,Maintenance of Synapse Structure (GO:0099558),0.03583154,31.59048,R270X D46 vs WT D46,GO_Biological_Process_2025
6,Tetrahydrofolate Metabolic Process (GO:0046653),0.04458920,24.56790,R270X D46 vs WT D46,GO_Biological_Process_2025


In [35]:
# Generate dot plot
if (nrow(all_data) > 0) {
  
  dot_plot <- ggplot(all_data, aes(x = Module, y = Term, size = Odds.Ratio, fill = P.value)) +
    geom_point(shape = 21, alpha = 0.8) +
    scale_fill_viridis(name = 'P-value', direction = -1) +
    scale_size_continuous(name = 'Odds Ratio', range = c(1, 8)) +
    xlab('') + ylab('') +
    labs(title = 'Significant GO Terms') +
    theme_minimal() +
    theme(
      panel.background = element_rect(fill = "white", color = NA),
      panel.grid = element_blank(),
      axis.line.x = element_line(color = "black"),
      axis.line.y = element_line(color = "black"),
      axis.ticks = element_line(color = "black"),
      axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5),
      axis.text.y = element_text(size = 9),
      legend.position = "right"
    )
  
  # Save the plot
  ggsave(filename = "06_DEG_GO/combined_DEG_timepoint_dot_plot.pdf", 
         plot = dot_plot, height = 6, width = 8)
}